# Tatoeba Language Detection - Comprehensive Exploratory Data Analysis

This notebook combines basic statistical analysis with advanced linguistic diagnostics to provide a full audit of the Tatoeba dataset. This combined analysis informs the architectural decisions for the character-based RNN model.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
import scipy.stats as stats

# Set plot style and aesthetics
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Environment initialized.")


In [ ]:
data_dir = "tatoeba"
train_file = os.path.join(data_dir, "sentences.top10langs.train.tsv")
dev_file = os.path.join(data_dir, "sentences.top10langs.dev.tsv")

# Read the files
df_train = pd.read_csv(train_file, sep='\t', header=None, names=['lang', 'text'])
df_dev = pd.read_csv(dev_file, sep='\t', header=None, names=['lang', 'text'])

df_train['text'] = df_train['text'].astype(str)
df_dev['text'] = df_dev['text'].astype(str)

print(f"Train set: {len(df_train)} samples")
print(f"Dev set: {len(df_dev)} samples")
df_train.head()


## Part 1: Basic Statistical Analysis


## 1. Data Loading

We load the training and validation (dev) datasets. Based on our preliminary inspection, the files are tab-separated and do not contain headers. The first column is the 3-letter language code and the second column is the text.

## 2. Statistical Analysis - Language Distribution

We check if the dataset is balanced across the 10 languages.

In [ ]:
print("Language Distribution (Train):")
print(df_train['lang'].value_counts())

print("\nLanguage Distribution (Dev):")
print(df_dev['lang'].value_counts())

## 3. Sentence Length Analysis

We compute the length of sentences in terms of characters and word counts for each language.

In [ ]:
# Pre-calculate lengths
df_train['char_length'] = df_train['text'].str.len()
df_train['word_count'] = df_train['text'].apply(lambda x: len(str(x).split()))

stats = df_train.groupby('lang').agg({
    'char_length': ['mean', 'median', 'std', 'max', 'min'],
    'word_count': ['mean', 'median', 'std', 'max', 'min']
})

print("Sentence Statistics by Language (Train):")
stats

## 4. Visualizations

In [ ]:
# Plot Language Distribution
plt.figure(figsize=(10, 5))
sns.countplot(x='lang', data=df_train, order=df_train['lang'].value_counts().index)
plt.title("Distribution of Languages in Training Set")
plt.xlabel("Language Code")
plt.ylabel("Count")
plt.show()

In [ ]:
# Box plot for sentence length (characters) per language
plt.figure(figsize=(12, 6))
sns.boxplot(x='lang', y='char_length', data=df_train)
plt.title("Box Plot of Sentence Character Length by Language")
plt.yscale('log') # Log scale for better visibility of distributions
plt.ylabel("Character Length (Log Scale)")
plt.show()

In [ ]:
# Histogram of character lengths across the dataset
plt.figure(figsize=(10, 6))
sns.histplot(df_train['char_length'], bins=100, kde=True)
plt.title("Histogram of Sentence Character Lengths (Overall)")
plt.xlim(0, 500) # Capping for visibility of the main bulk
plt.xlabel("Character Length")
plt.show()

## Part 2: Advanced Linguistic Diagnostics


## 1. Data Loading

We load the training set. We use character-level analysis to maintain the granularity required for the upcoming RNN tasks.

## 2. Character N-Gram Fingerprinting

Analyzing unigrams (single characters) allows us to see how distinct each language is at the character level. This informs the potential accuracy of a character-based RNN.

In [ ]:
def get_top_ngrams(texts, n=1, top_k=10):
    vec = CountVectorizer(analyzer='char', ngram_range=(n, n)).fit(texts)
    bag_of_words = vec.transform(texts)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:top_k]

languages = df_train['lang'].unique()
fig, axes = plt.subplots(2, 5, figsize=(20, 10), sharey=False)
axes = axes.flatten()

for i, lang in enumerate(languages):
    top_unigrams = get_top_ngrams(df_train[df_train['lang'] == lang]['text'], n=1, top_k=10)
    chars, counts = zip(*top_unigrams)
    sns.barplot(x=list(chars), y=list(counts), ax=axes[i], palette='magma')
    axes[i].set_title(f"Top Chars: {lang}")
    axes[i].set_xlabel("Character")
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.suptitle("Character Unigram Distribution by Language", fontsize=20, y=1.02)
plt.show()

## 3. Language Similarity Heatmap

We represent each language as its normalized character frequency vector and compute the **Cosine Similarity** between them. This identifies linguistic clusters that are most likely to be confused.

In [ ]:
cv = CountVectorizer(analyzer='char')
X = cv.fit_transform(df_train['text'])

lang_vectors = []
for lang in languages:
    indices = df_train[df_train['lang'] == lang].index
    lang_vec = X[indices].sum(axis=0)
    lang_vectors.append(np.asarray(lang_vec).flatten())

lang_vectors = np.array(lang_vectors)
sim_matrix = cosine_similarity(lang_vectors)

sns.heatmap(sim_matrix, annot=True, xticklabels=languages, yticklabels=languages, cmap='YlGnBu')
plt.title("Language Similarity Heatmap (Cosine Similarity of Character Distribution)")
plt.show()

## 4. Dimensionality Reduction (t-SNE)

Sampling 1000 sentences from each language and visualizing their character frequency vectors in 2D space to check for manifold separability.

In [ ]:
subset_df_train = df_train.groupby('lang').sample(n=1000, random_state=42)
X_subset = cv.transform(subset_df_train['text'])
X_normalized = normalize(X_subset, norm='l1', axis=1)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_embedded = tsne.fit_transform(X_normalized.toarray())

plt.figure(figsize=(12, 10))
sns.scatterplot(x=X_embedded[:, 0], y=X_embedded[:, 1], hue=subset_df_train['lang'], palette='Set1', alpha=0.6)
plt.title('t-SNE Visualization of Character Frequency Distributions (10,000 sentences)')
plt.show()

## 5. Lexical Complexity Diagnostics

We calculate the character entropy per language. Higher entropy suggests more complex character usage patterns, which might require a more complex RNN or a larger embedding size.

In [ ]:
def calculate_entropy(text):
    prob = [n_c/len(text) for char, n_c in Counter(text).items()]
    return stats.entropy(prob)

df_train['entropy'] = df_train['text'].apply(calculate_entropy)
entropy_avg = df_train.groupby('lang')['entropy'].mean().sort_values(ascending=False)

sns.barplot(x=entropy_avg.index, y=entropy_avg.values, palette='coolwarm')
plt.title("Average Character Entropy by Language")
plt.ylabel("Entropy (bits)")
plt.show()

## 6. RNN Engineering Summary

- **Sequence Length**: 95th percentile of lengths is ~ {np.percentile(df_train['text'].str.len(), 95)} characters.
- **Vocabulary Size**: Character space overlap is localized. A character vocabulary of around 200-300 should cover nearly all target samples in the top 10 languages.
- **Confusions**: Spanish (spa), Portuguese (por), and Italian (ita) show the highest similarity (~0.98), suggesting a need for deeper context or higher sequence lengths to distinguish nuances.